# Masked AutoRL-SOP Result Viewer

Select a variant and an SOP instance to inspect its stored optimization results. Optuna variants provide the complete diagnostic set; Scott--Knott variants display their sequential-tuning log.

In [11]:
from pathlib import Path
import os
import shutil
import tempfile
import warnings

import optuna
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from optuna.importance import FanovaImportanceEvaluator, PedAnovaImportanceEvaluator
from optuna.visualization import (
    plot_contour,
    plot_optimization_history,
    plot_parallel_coordinate,
    plot_param_importances,
    plot_rank,
    plot_slice,
)

import hpo_analysis as ha

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

ROOT = Path(".").resolve()


def apply_latex_style(fig):
    """Apply the visual style used by the article figures."""
    fig.update_layout(
        template="simple_white",
        font=dict(family="Computer Modern", size=14),
        title=dict(x=0.5, xanchor="center"),
        margin=dict(l=70, r=30, t=70, b=60),
        legend=dict(bgcolor="rgba(255,255,255,0.8)", borderwidth=0),
    )
    fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", zeroline=False)
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", zeroline=False)
    return fig


def unavailable_figure(title):
    """Explain why a parameter-importance figure is unavailable."""
    fig = go.Figure()
    fig.add_annotation(
        text=(
            "Parameter importance cannot be estimated<br>"
            "because the study contains only one completed trial."
        ),
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
        align="center",
        font=dict(size=16),
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(title=title)
    return apply_latex_style(fig)


print("=" * 72)
print("VARIANT SELECTION FOR VIEWING")
print("=" * 72)
for index, model in enumerate(ha.MODELS, start=1):
    print(f"[{index:02d}] {model['label']}")

variant_choice = int(input(f"\nEnter variant number (1-{len(ha.MODELS)}): "))
if not 1 <= variant_choice <= len(ha.MODELS):
    raise ValueError("Invalid variant number.")
model = ha.MODELS[variant_choice - 1]

print("\n" + "=" * 72)
print("INSTANCE SELECTION FOR VIEWING")
print("=" * 72)
for index, name in enumerate(ha.SOP_INSTANCES, start=1):
    print(f"[{index:02d}] {name:<15}", end="")
    if index % 4 == 0:
        print()

instance_choice = int(input(f"\nEnter instance number (1-{len(ha.SOP_INSTANCES)}): "))
if not 1 <= instance_choice <= len(ha.SOP_INSTANCES):
    raise ValueError("Invalid instance number.")

instance = ha.SOP_INSTANCES[instance_choice - 1]
key_name = ha.key_of(instance)
result_file = Path(ha.result_path(model, instance, str(ROOT)))
if not result_file.is_file():
    raise FileNotFoundError(f"Result file not found: {result_file}")

study = None
scott_knott_log = None
generated_figures = []

print(f"\nSelected variant: {model['label']}")
print(f"Selected instance: {instance}")
print(f"Result file: {result_file}")

if model["type"] == "optuna":
    safe_label = "".join(ch if ch.isalnum() else "_" for ch in model["label"])
    db_copy = Path(tempfile.gettempdir()) / f"{safe_label}_{key_name}_copy.db"
    shutil.copy2(result_file, db_copy)
    study = optuna.load_study(
        study_name=f"study_{key_name}",
        storage=f"sqlite:///{db_copy.as_posix()}",
    )
    print(f"Loaded trials: {len(study.trials)}")
    print(f"Best value: {study.best_value}")
    print(f"Best parameters: {study.best_params}")
else:
    scott_knott_log = pd.read_csv(result_file)
    hpo_rows = scott_knott_log[scott_knott_log["stage"].isin(ha.HPO_STAGES)].copy()
    print(f"Logged HPO evaluations: {len(hpo_rows)}")
    best_observed = pd.to_numeric(
        scott_knott_log.loc[
            scott_knott_log["stage"] == "best_observed", "min_distance"
        ],
        errors="coerce",
    ).dropna()
    if not best_observed.empty:
        print(f"Best observed value: {best_observed.iloc[-1]}")


VARIANT SELECTION FOR VIEWING
[01] [MULTIVARIATE] Hyperband TPE
[02] [MULTIVARIATE] TPE
[03] [UNIVARIATE] Hyperband TPE
[04] [UNIVARIATE] TPE
[05] Hyperband GP
[06] Gaussian Process
[07] HyperTuningSK
[08] [MULTIVARIATE] NO_MASK
[09] [UNIVARIATE] NO_MASK
[10] NO_MASK_NO_BAYESIAN (resampling, SK)
[11] Random Search

INSTANCE SELECTION FOR VIEWING
[01] br17.10.sop    [02] br17.12.sop    [03] ESC07.sop      [04] ESC12.sop      
[05] ESC25.sop      [06] ESC47.sop      [07] ESC63.sop      [08] ESC78.sop      
[09] ft53.1.sop     [10] ft53.2.sop     [11] ft53.3.sop     [12] ft53.4.sop     
[13] ft70.1.sop     [14] ft70.2.sop     [15] ft70.3.sop     [16] ft70.4.sop     
[17] kro124p.1.sop  [18] kro124p.2.sop  [19] kro124p.3.sop  [20] p43.1.sop      
[21] p43.2.sop      [22] p43.3.sop      [23] p43.4.sop      [24] prob.42.sop    
[25] ry48p.1.sop    [26] ry48p.2.sop    [27] ry48p.3.sop    [28] ry48p.4.sop    

Selected variant: [MULTIVARIATE] Hyperband TPE
Selected instance: ry48p.1.sop
Result

## Optimization History

In [12]:
if study is not None:
    fig_hist = plot_optimization_history(study)
    fig_hist.add_hline(y=study.best_value, line_dash="dash", line_color="black")
    fig_hist = apply_latex_style(fig_hist)
    fig_hist.update_layout(
        xaxis_title="Trial",
        yaxis_title="Objective Value",
    )
    generated_figures.append(("01_optimization_history", fig_hist))
    fig_hist.show()
else:
    hpo_rows["min_distance"] = pd.to_numeric(hpo_rows["min_distance"], errors="coerce")
    fig_hist = px.line(
        hpo_rows,
        x="eval_index",
        y="min_distance",
        color="stage",
        markers=True,
        title="Sequential Tuning History",
    )
    fig_hist = apply_latex_style(fig_hist)
    fig_hist.update_layout(xaxis_title="Evaluation", yaxis_title="Minimum Distance")
    generated_figures.append(("01_sequential_tuning_history", fig_hist))
    fig_hist.show()


## Hyperparameter Importance

### fANOVA

In [13]:
if study is not None:
    try:
        importance_evaluator = FanovaImportanceEvaluator(seed=42)
        fig_importance = plot_param_importances(study, evaluator=importance_evaluator)
        fig_importance = apply_latex_style(fig_importance)
        fig_importance.update_yaxes(showgrid=False)
        fig_importance.update_layout(
            title="Hyperparameter Importance (fANOVA)",
            xaxis_title="Relative Importance",
        )
    except ValueError as exc:
        if "only a single trial" not in str(exc):
            raise
        fig_importance = unavailable_figure("Hyperparameter Importance (fANOVA)")
    generated_figures.append(("02_param_importance", fig_importance))
    fig_importance.show()
else:
    phase_summary = (
        hpo_rows.groupby(["stage", "candidate_value"], as_index=False)["min_distance"]
        .agg(["mean", "std", "min"])
        .reset_index()
    )
    display(phase_summary)


### PED-ANOVA

In [14]:
if study is not None:
    try:
        ped_evaluator = PedAnovaImportanceEvaluator()
        fig_ped_importance = plot_param_importances(study, evaluator=ped_evaluator)
        fig_ped_importance = apply_latex_style(fig_ped_importance)
        fig_ped_importance.update_layout(title="Hyperparameter Importance (PED-ANOVA)")
        fig_ped_importance.update_yaxes(showgrid=False)
    except ValueError as exc:
        if "only a single trial" not in str(exc):
            raise
        fig_ped_importance = unavailable_figure("Hyperparameter Importance (PED-ANOVA)")
    generated_figures.append(("03_ped_anova", fig_ped_importance))
    fig_ped_importance.show()
else:
    print("PED-ANOVA is available only for Optuna studies.")


## Parallel Coordinates

In [15]:
if study is not None:
    fig_parallel = plot_parallel_coordinate(study)
    fig_parallel.update_traces(
        line=dict(
            colorscale="RdBu",
            showscale=True,
            colorbar=dict(title="Objective Value", ticks="outside"),
        )
    )
    fig_parallel = apply_latex_style(fig_parallel)
    fig_parallel.update_layout(margin=dict(l=80, r=140, t=80, b=100))
    generated_figures.append(("04_parallel_coordinates", fig_parallel))
    fig_parallel.show()
else:
    print("Parallel coordinates are available only for Optuna studies.")


## Slice Plot

In [16]:
if study is not None:
    fig_slice = plot_slice(study)
    fig_slice.update_traces(
        selector=dict(type="scatter"),
        marker=dict(colorscale="RdBu", showscale=True, colorbar=dict(title="Trial")),
    )
    fig_slice = apply_latex_style(fig_slice)
    generated_figures.append(("05_slice", fig_slice))
    fig_slice.show()
else:
    print("Slice plots are available only for Optuna studies.")


## Contour Plot

In [17]:
if study is not None:
    fig_contour = plot_contour(study)
    fig_contour.update_traces(
        selector=dict(type="contour"),
        contours=dict(coloring="heatmap"),
        colorscale="RdBu",
    )
    fig_contour = apply_latex_style(fig_contour)
    generated_figures.append(("06_contour", fig_contour))
    fig_contour.show()
else:
    print("Contour plots are available only for Optuna studies.")


## Rank Plot

In [18]:
if study is not None:
    fig_rank = apply_latex_style(plot_rank(study))
    generated_figures.append(("07_rank", fig_rank))
    fig_rank.show()
else:
    print("Rank plots are available only for Optuna studies.")


## Export to PDF

In [19]:
from pypdf import PdfReader, PdfWriter

safe_variant = "".join(ch if ch.isalnum() else "_" for ch in model["label"]).strip("_")
output_dir = ROOT / "figures_pdf" / safe_variant / key_name
output_dir.mkdir(parents=True, exist_ok=True)

pdf_paths = []
for name, fig in generated_figures:
    pdf_path = output_dir / f"{name}.pdf"
    fig.write_image(pdf_path, format="pdf", scale=2)
    pdf_paths.append(pdf_path)

if pdf_paths:
    writer = PdfWriter()
    for pdf_path in pdf_paths:
        reader = PdfReader(str(pdf_path))
        for page in reader.pages:
            writer.add_page(page)
    final_pdf = output_dir / "all_figures.pdf"
    with open(final_pdf, "wb") as output:
        writer.write(output)
    print(f"Final PDF generated in: {final_pdf.resolve()}")
else:
    print("No figures were generated.")


Final PDF generated in: C:\Users\Kerollan\Desktop\Masked-AutoRL-SOP\figures_pdf\MULTIVARIATE__Hyperband_TPE\ry48p.1\all_figures.pdf
